# XJTU cycle-100 EOL timing

Estimate remaining useful life (RUL) at the end of cycle 100 and measure how it changes the late-life state-of-health (SOH) forecast for 3C cells.

Run all cells in the project Python environment. The notebook reads the existing Notebook 06–08 reports and the XJTU data under Dataset. It reads one cell at a time and caches compact features under reports/lifeguard_eol_timing_results. The results include a summary, per-cell forecasts and scores, quality decisions, and a ZIP archive.

The charge summaries and the two curve variants share the same timing cutoff, training folds, and SOH initializer. The discharge variant has access to the completed discharge through cycle 100. Reported results describe this XJTU cohort.

In [1]:
from collections import defaultdict
from contextlib import contextmanager
import csv
from io import TextIOWrapper
import json
from pathlib import Path, PurePosixPath
import pickle
from statistics import mean, median
from zipfile import ZIP_DEFLATED, ZipFile

import numpy as np
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

DATA_PATH = None  # Optional XJTU directory or XJTU.zip.
OUTPUT_ROOT = None  # Defaults to the project reports folder.

ANCHOR = 100
RATE = 3
EOL_SOH = 80.0
MAX_STEPS = 600
TAIL = 20
WINDOWS = ((41, 50), (91, 100))
MIN_TRACES = 5
ALPHAS = (0.1, 1.0, 10.0, 100.0)
HORIZONS = (1, 10, 50, 100, 150)
HORIZON_TOLERANCE = 0.05
CHARGE_GRID = np.linspace(3.8, 4.1, 31)
DISCHARGE_GRID = np.linspace(4.0, 3.0, 51)
DAMAGE_GRID = np.linspace(0, 1, 101)
FEATURE_VERSION = 1
SUMMARY_FIELDS = ("recent_charge_ah", "recent_4_0_to_4_1_s", "recent_taper_s")
CHARGE_FIELD = "charge_curve_log_variance"
DISCHARGE_FIELD = "discharge_curve_log_variance"
NB06 = "lifeguard_rul_charge_features_results"
NB07 = "lifeguard_digital_twin_validation_results"
NB08 = "lifeguard_fixed_anchor_soh_results"
OUTPUT_NAME = "lifeguard_eol_timing_results"
PRIOR = "nb06_pooled_ridge"
RECENT = "nb06_recent_ridge"
ENSEMBLE = "nb06_ridge_average"
STATE = "soh_dependent"
INCUMBENT = "late_life_calibrated"
SUMMARY = "eol_timing_charge_summary"
CHARGE = "eol_timing_charge_curve"
DISCHARGE = "eol_timing_charge_discharge"
MODEL_FIELDS = {
    SUMMARY: SUMMARY_FIELDS,
    CHARGE: (*SUMMARY_FIELDS, CHARGE_FIELD),
    DISCHARGE: (*SUMMARY_FIELDS, CHARGE_FIELD, DISCHARGE_FIELD),
}

cwd = Path.cwd()
project = next((folder for folder in (cwd, *cwd.parents)
                if (folder / "README_LifeGuard_RL.md").is_file() and (folder / "reports").is_dir()), None)
if project is None:
    raise FileNotFoundError("Open the notebook from the LifeGuard-RL project.")
reports = project / "reports"
output = (Path(OUTPUT_ROOT).expanduser().resolve() if OUTPUT_ROOT else reports) / OUTPUT_NAME


@contextmanager
def report_rows(report, table):
    folder = reports / report
    if (folder / table).is_file():
        with (folder / table).open(encoding="utf-8", newline="") as file:
            yield csv.DictReader(file)
        return
    archive_path = folder.with_suffix(".zip")
    with ZipFile(archive_path) as archive:
        matches = [name for name in archive.namelist() if PurePosixPath(name).name == table]
        if len(matches) != 1:
            raise ValueError(f"Expected one {table} in {archive_path.name}, found {len(matches)}")
        with TextIOWrapper(archive.open(matches[0]), encoding="utf-8", newline="") as file:
            yield csv.DictReader(file)


def write_csv(path, rows, fields=None):
    if fields is None:
        fields = list(rows[0])
    temp = path.with_suffix(path.suffix + ".tmp")
    with temp.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    temp.replace(path)


def at_anchor(row):
    return int(row["landmark_cycle"]) == ANCHOR and int(row["charge_rate_c"]) == RATE

In [2]:
def load_baselines():
    with report_rows(NB06, "landmark_features.csv") as rows:
        summaries = {
            row["cell_id"]: {key: float(row[key]) for key in SUMMARY_FIELDS}
            for row in rows if at_anchor(row)
        }
    with report_rows(NB07, "eol_audit.csv") as rows:
        endpoints = {
            row["cell_id"]: int(row["observed_eol_cycle"])
            for row in rows if int(row["charge_rate_c"]) == RATE and row["event_status"] == "observed"
        }
    if set(summaries) != set(endpoints) or any(cycle <= ANCHOR for cycle in endpoints.values()):
        raise ValueError("The Notebook 06 summaries must cover the verified 3C cycle-100 cohort.")

    with report_rows(NB07, "cell_splits.csv") as rows:
        splits = {(int(row["fold"]), row["role"], row["cell_id"])
                  for row in rows if int(row["charge_rate_c"]) == RATE}
    with report_rows(NB06, "cell_splits.csv") as rows:
        reference = {(int(row["fold"]), row["role"], row["cell_id"]) for row in rows if at_anchor(row)}
    if splits != reference:
        raise ValueError("The Notebook 06 and 07 cell splits differ.")
    test_folds = {cell: fold for fold, role, cell in splits if role == "test"}
    training = {fold: sorted(cell for f, role, cell in splits if f == fold and role == "train")
                for fold in sorted(set(test_folds.values()))}
    if len(training) != 5 or set(test_folds) != set(endpoints):
        raise ValueError("Expected the five saved 3C folds.")
    for fold, group in training.items():
        if set(group) != {cell for cell in endpoints if test_folds[cell] != fold}:
            raise ValueError(f"Incomplete training membership in fold {fold}")

    names = {"ridge_charge_history_pooled": PRIOR, "ridge_3c_recent_signals": RECENT}
    direct = {PRIOR: {}, RECENT: {}}
    with report_rows(NB06, "rul_predictions.csv") as rows:
        for row in rows:
            if at_anchor(row) and row["model"] in names:
                cell = row["cell_id"]
                if int(row["fold"]) != test_folds[cell] or int(row["observed_eol_cycle"]) != endpoints[cell]:
                    raise ValueError(f"RUL reference does not match the split or target for {cell}")
                direct[names[row["model"]]][cell] = float(row["predicted_rul_cycles"])
    if any(set(group) != set(endpoints) for group in direct.values()):
        raise ValueError("Missing a held-out prediction from the Notebook 06 reference.")
    direct[ENSEMBLE] = {cell: (direct[PRIOR][cell] + direct[RECENT][cell]) / 2 for cell in endpoints}

    starts = {}
    with report_rows(NB07, "rollout_starts.csv") as rows:
        for row in rows:
            if at_anchor(row) and row["initializer"] == "charge" and row["model"] == STATE:
                cell = row["cell_id"]
                if int(row["fold"]) != test_folds[cell]:
                    raise ValueError(f"Notebook 07 rollout fold mismatch for {cell}")
                starts[cell] = float(row["initial_soh_pct"]), int(float(row["predicted_rul_cycles"]))

    calibration = defaultdict(list)
    with report_rows(NB07, "transition_calibration.csv") as rows:
        for row in rows:
            if int(row["charge_rate_c"]) == RATE and row["used"] == "True":
                calibration[int(row["fold"])].append(
                    (float(row["bin_bottom_soh_pct"]) + 1.0, float(row["model_fade_pp_per_cycle"]))
                )
    calibration = {fold: tuple(zip(*sorted(points))) for fold, points in calibration.items()}

    def read_trajectories(report, model):
        recorded, predicted = defaultdict(dict), defaultdict(dict)
        with report_rows(report, "soh_trajectories.csv") as rows:
            for row in rows:
                if at_anchor(row) and row["initializer"] == "charge" and row["model"] == model:
                    cell, cycle = row["cell_id"], int(row["cycle"])
                    recorded[cell][cycle] = float(row["observed_soh_pct"])
                    predicted[cell][cycle] = float(row["predicted_soh_pct"])
        return recorded, predicted

    observed, baseline = read_trajectories(NB07, STATE)
    aligned_observed, incumbent = read_trajectories(NB08, INCUMBENT)
    if observed != aligned_observed:
        raise ValueError("The Notebook 07 and 08 evaluation timelines differ.")
    for cell, end in endpoints.items():
        if set(observed[cell]) != set(range(ANCHOR, end + 1)):
            raise ValueError(f"Incomplete measured trajectory for {cell}")

    return summaries, endpoints, test_folds, training, direct, starts, calibration, observed, baseline, incumbent


summaries, endpoints, test_folds, training, direct, starts, calibration, observed, baseline, incumbent = load_baselines()
cells = sorted(endpoints, key=lambda cell: int(cell.rsplit("-", 1)[1]))
print(f"Loaded {len(cells)} verified 3C cells and {len(training)} saved folds.")

Loaded 15 verified 3C cells and 5 saved folds.


## Curve features through cycle 100

Use a common voltage range in the charge and discharge segments. Convert current and time to cumulative capacity, then express each curve as a fraction of nominal capacity. Compute the median curve in cycles 41–50 and 91–100, and use the log variance of the difference as one feature for each direction. A window needs five valid traces. The charge-only model uses the charge segment.

The cache is keyed by the feature version and raw cell file. As soon as a cell is processed its features are saved.

In [3]:
class XJTUUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        raise pickle.UnpicklingError(f"Unexpected global in XJTU data: {module}.{name}")


def find_data():
    if DATA_PATH is not None:
        return Path(DATA_PATH).expanduser().resolve(strict=True)
    folder = project / "Dataset" / "XJTU"
    for path in (folder, folder.with_suffix(".zip")):
        if path.exists():
            return path
    raise FileNotFoundError("Set DATA_PATH to the XJTU cell directory or XJTU.zip.")


@contextmanager
def cell_files(path):
    if path.is_dir():
        def find(cell):
            file = path / f"{cell}.pkl"
            status = file.stat()
            return f"v{FEATURE_VERSION}:{status.st_size}:{status.st_mtime_ns}", lambda: file.open("rb")

        yield find
        return

    with ZipFile(path) as archive:
        entries = {PurePosixPath(info.filename).stem: info for info in archive.infolist()
                   if PurePosixPath(info.filename).suffix == ".pkl"}

        def find(cell):
            member = entries[cell]
            return f"v{FEATURE_VERSION}:{member.file_size}:{member.CRC}", lambda: archive.open(member)

        yield find


def capacity_on_grid(time, voltage, current, direction, grid):
    levels = direction * grid
    active = direction * current
    starts = np.flatnonzero(active > 1.0)
    if not starts.size:
        return None
    start = starts[0]
    stops = np.flatnonzero(active[start:] <= 0.1)
    end = start + stops[0] if stops.size else len(active)
    t, v, amps = time[start:end], voltage[start:end] * direction, active[start:end]
    if len(t) < 2 or np.any(np.diff(t) <= 0):
        return None

    high = np.searchsorted(np.maximum.accumulate(v), levels)
    if np.any(high == 0) or np.any(high == len(t)):
        return None
    low = high - 1
    voltage_step = v[high] - v[low]
    if np.any(voltage_step <= 0):
        return None
    position = (levels - v[low]) / voltage_step
    dt = np.diff(t)
    cumulative = np.r_[0.0, np.cumsum(dt * (amps[:-1] + amps[1:]) / 7200)]
    crossing_amps = amps[low] + position * (amps[high] - amps[low])
    charge = cumulative[low] + dt[low] * position * (amps[low] + crossing_amps) / 7200
    return charge - charge[0]


def extract_features(stream, cell):
    battery = XJTUUnpickler(stream).load()
    capacity = float(battery["nominal_capacity_in_Ah"])
    if (battery["cell_id"] != cell or capacity != 2.0
            or [step["rate_in_C"] for step in battery["charge_protocol"]] != [RATE]
            or [step["rate_in_C"] for step in battery["discharge_protocol"]] != [1.0]):
        raise ValueError(f"Unexpected identity, nominal capacity, or cycling protocol in {cell}.")

    directions = ((CHARGE_FIELD, 1, CHARGE_GRID), (DISCHARGE_FIELD, -1, DISCHARGE_GRID))
    traces = {(name, window): [] for name, _, _ in directions for window in WINDOWS}
    found = set()
    for cycle in battery["cycle_data"]:
        number = cycle["cycle_number"]
        if number > ANCHOR:
            break
        window = next((pair for pair in WINDOWS if pair[0] <= number <= pair[1]), None)
        if window is None or cycle.get("attribute") != "Cycling":
            continue
        if number in found:
            raise ValueError(f"Repeated cycle {number} in {cell}")
        found.add(number)

        arrays = tuple(np.asarray(cycle[field], dtype=float)
                       for field in ("time_in_s", "voltage_in_V", "current_in_A"))
        if any(array.ndim != 1 or array.size != arrays[0].size or not np.isfinite(array).all()
               for array in arrays):
            continue
        for name, direction, grid in directions:
            curve = capacity_on_grid(*arrays, direction, grid)
            if curve is not None:
                traces[name, window].append(curve / capacity)

    features = {}
    for name, _, _ in directions:
        early, recent = (traces[name, window] for window in WINDOWS)
        value = None
        if min(len(early), len(recent)) >= MIN_TRACES:
            change = np.median(recent, axis=0) - np.median(early, axis=0)
            value = float(np.log(max(float(np.var(change)), 1e-12)))
        features[name] = value
        features[f"{name}_early_count"] = len(early)
        features[f"{name}_recent_count"] = len(recent)
    return features


def load_curve_features(path):
    output.mkdir(parents=True, exist_ok=True)
    cache = output / "curve_features.csv"
    records = {}
    if cache.is_file():
        with cache.open(encoding="utf-8", newline="") as file:
            records = {row["cell_id"]: row for row in csv.DictReader(file) if row["cell_id"] in cells}
    with cell_files(path) as locate:
        for cell in cells:
            key, open_file = locate(cell)
            cached = records.get(cell, {})
            if cached.get("source_key") == key and all(field in cached for field in (CHARGE_FIELD, DISCHARGE_FIELD)):
                continue
            with open_file() as stream:
                features = extract_features(stream, cell)
            records[cell] = {"cell_id": cell, "source_key": key, **features}
            write_csv(cache, [records[cell] for cell in cells if cell in records])
            print(f"Cached {cell}")

    return {cell: {field: float(records[cell][field]) if records[cell][field] not in (None, "") else np.nan
                   for field in (CHARGE_FIELD, DISCHARGE_FIELD)}
            for cell in cells}


data_path = find_data()
with threadpool_limits(limits=1):
    curve_features = load_curve_features(data_path)
features = {cell: {**summaries[cell], **curve_features[cell]} for cell in cells}
print("Complete charge curve features:", sum(np.isfinite(row[CHARGE_FIELD]) for row in features.values()))
print("Complete discharge curve features:", sum(np.isfinite(row[DISCHARGE_FIELD]) for row in features.values()))

Complete charge curve features: 15
Complete discharge curve features: 15


In [4]:
def matrix(group, fields):
    return np.array([[features[cell][field] for field in fields] for cell in group], dtype=float)


def target(group):
    return np.array([endpoints[cell] - ANCHOR for cell in group], dtype=float)


def fit_ridge(x, y, alpha):
    return make_pipeline(StandardScaler(), Ridge(alpha=alpha)).fit(x, y)


def fit_and_predict(train_cells, test_cells, fields):
    train = [cell for cell in train_cells if np.isfinite(matrix([cell], fields)).all()]
    test = [cell for cell in test_cells if np.isfinite(matrix([cell], fields)).all()]
    if len(train) < 3 or not test:
        return {}, None

    x, y = matrix(train, fields), target(train)
    candidates = []
    for alpha in ALPHAS:
        errors = []
        for i in range(len(train)):
            keep = np.arange(len(train)) != i
            model = fit_ridge(x[keep], y[keep], alpha)
            value = max(float(model.predict(x[i:i + 1])[0]), 0.0)
            errors.append(abs(value - y[i]))
        candidates.append((mean(errors), -alpha))
    alpha = -min(candidates)[1]
    model = fit_ridge(x, y, alpha)
    return dict(zip(test, np.maximum(model.predict(matrix(test, fields)), 0.0).tolist())), alpha


def forecast_rul():
    forecasts = {name: {} for name in MODEL_FIELDS}
    methods = {name: {} for name in MODEL_FIELDS}
    tuning = []
    for fold, group in training.items():
        test = [cell for cell in cells if test_folds[cell] == fold]
        for name, fields in MODEL_FIELDS.items():
            estimated, alpha = fit_and_predict(group, test, fields)
            if name == SUMMARY and set(estimated) != set(test):
                raise ValueError(f"Missing required charge summaries in fold {fold}")
            for cell in test:
                if cell in estimated:
                    forecasts[name][cell] = estimated[cell]
                    methods[name][cell] = "fitted"
                else:
                    forecasts[name][cell] = forecasts[SUMMARY][cell]
                    methods[name][cell] = "charge_summary_fallback"
            tuning.append({"fold": fold, "model": name, "ridge_alpha": alpha,
                           "train_cells_with_features": int(np.isfinite(matrix(group, fields)).all(axis=1).sum()),
                           "test_cells": len(test), "test_cells_with_features": len(estimated)})
    return forecasts, methods, tuning


with threadpool_limits(limits=1):
    fitted_rul, methods, tuning = forecast_rul()
direct.update(fitted_rul)
print(f"Produced {len(cells)} cycle-100 RUL predictions per model.")

Produced 15 cycle-100 RUL predictions per model.


In [5]:
def first_crossing(curve):
    steps = np.flatnonzero(curve <= EOL_SOH)
    return int(steps[0]) if steps.size else None


def baseline_curve(initial, profile):
    grid, rates = profile
    values = [initial]
    for _ in range(MAX_STEPS):
        previous = values[-1]
        values.append(max(EOL_SOH, previous - np.interp(previous, grid, rates)))
    return np.asarray(values)


def fit_decay_shape(fold):
    templates = []
    for cell in training[fold]:
        trace = observed[cell]
        start, end = ANCHOR, endpoints[cell]
        if trace[start] <= EOL_SOH:
            continue
        cycles = np.arange(start, end + 1)
        values = np.array([trace[c] for c in cycles])
        aligned = np.interp(start + DAMAGE_GRID * (end - start), cycles, values)
        damage = np.clip((values[0] - aligned) / (values[0] - EOL_SOH), 0, 1 - 1e-9)
        damage = np.maximum.accumulate(damage)
        damage[0], damage[-1] = 0.0, 1.0
        templates.append(damage)
    if len(templates) < 2:
        raise ValueError(f"Insufficient completed training curves in fold {fold}")
    return np.median(templates, axis=0)


def calibrate_timing(original, earlier_rul, shape):
    old_rul = first_crossing(original)
    if old_rul is None or old_rul <= 0 or earlier_rul < 0 or not np.isfinite(earlier_rul):
        raise ValueError("Timing calibration needs a valid cycle-100 forecast and an earlier RUL estimate.")
    duration = (old_rul + earlier_rul) / 2
    transition = min(50, old_rul // 2, int(duration // 2))
    start_damage = np.interp(transition / duration, DAMAGE_GRID, shape)
    steps = np.arange(transition + 1, len(original))
    damage = np.interp(steps / duration, DAMAGE_GRID, shape)
    remaining = (1 - damage) / (1 - start_damage)
    values = original.copy()
    values[steps] = np.where(steps >= duration, EOL_SOH,
                             EOL_SOH + (original[transition] - EOL_SOH) * remaining)
    expected = int(np.ceil(duration)) if duration <= MAX_STEPS else None
    if first_crossing(values) != expected or np.any(np.diff(values) > 1e-9):
        raise AssertionError("The revised curve must be monotone and cross at the predicted cycle.")
    return values, transition, duration


templates = {fold: fit_decay_shape(fold) for fold in training}
rollouts = {name: {} for name in direct}
for cell in cells:
    fold = test_folds[cell]
    initial, earlier_crossing = starts[cell]
    original = baseline_curve(initial, calibration[fold])
    timeline = sorted(observed[cell])
    offsets = np.array(timeline) - ANCHOR
    if (first_crossing(original) != earlier_crossing
            or not np.allclose(original[offsets], [baseline[cell][t] for t in timeline], rtol=0, atol=1e-9)):
        raise ValueError(f"Notebook 07 baseline forecast differs for {cell}")
    for name, predictions in direct.items():
        values, transition, duration = calibrate_timing(original, predictions[cell], templates[fold])
        if name == PRIOR and not np.allclose(values[offsets], [incumbent[cell][t] for t in timeline], rtol=0, atol=1e-9):
            raise ValueError(f"Notebook 08 incumbent forecast differs for {cell}")
        rollouts[name][cell] = {"values": values, "gate": transition, "duration": duration,
                                "rul": first_crossing(values)}
print("Reproduced the Notebook 07 and 08 curves before scoring the candidates.")

Reproduced the Notebook 07 and 08 curves before scoring the candidates.


In [6]:
CONTEXT = {"charge_rate_c": RATE, "landmark_cycle": ANCHOR, "initializer": "charge"}
MODEL_NAMES = {
    PRIOR: "Notebook 06 pooled Ridge",
    RECENT: "Notebook 06 recent-signal Ridge",
    ENSEMBLE: "Notebook 06 equal-weight average",
    STATE: "Notebook 07 SOH-dependent fade",
    INCUMBENT: "Notebook 08 late-life calibrated",
    SUMMARY: "Charge summary",
    CHARGE: "Charge summary + charge curve",
    DISCHARGE: "Charge summary + charge and discharge curves",
}


def rul_metrics(name, predictions):
    errors = {cell: float(prediction - (endpoints[cell] - ANCHOR))
              for cell, prediction in predictions.items() if prediction is not None}
    short = [abs(error) for cell, error in errors.items() if endpoints[cell] <= 200]
    return {
        **CONTEXT, "model": name, "observed_test_cells": len(cells),
        "predicted_test_cells": len(errors),
        "mae_cycles": mean(abs(error) for error in errors.values()) if errors else None,
        "worst_error_cycles": max(map(abs, errors.values()), default=None),
        "short_life_mae_cycles": mean(short) if short else None,
        "short_life_cells": len(short),
        "signed_bias_cycles": mean(errors.values()) if errors else None,
    }


def score_rollouts():
    evaluated = {STATE: baseline, INCUMBENT: incumbent}
    evaluated.update({
        name: {cell: {cycle: group[cell]["values"][cycle - ANCHOR] for cycle in observed[cell]}
               for cell in cells}
        for name, group in rollouts.items() if name != PRIOR
    })
    timings = {STATE: {cell: starts[cell][1] for cell in cells},
               INCUMBENT: {cell: rollouts[PRIOR][cell]["rul"] for cell in cells}}
    timings.update({name: {cell: item["rul"] for cell, item in group.items()}
                    for name, group in rollouts.items() if name != PRIOR})

    cell_rows, soh_rows = [], []
    for name, trajectories in evaluated.items():
        errors = {cell: {cycle: float(abs(trajectories[cell][cycle] - actual))
                         for cycle, actual in observed[cell].items() if cycle > ANCHOR}
                  for cell in cells}
        tails = {cell: [error for cycle, error in errors[cell].items() if cycle >= endpoints[cell] - TAIL + 1]
                 for cell in cells}
        windows = {
            "full_curve": {cell: list(errors[cell].values()) for cell in cells},
            "last_20_cycles": tails,
            "observed_eol_cycle": {cell: [errors[cell][endpoints[cell]]] for cell in cells},
        }
        for horizon in HORIZONS:
            cycle = ANCHOR + horizon
            windows[f"horizon_{horizon}"] = {
                cell: [errors[cell][cycle]] for cell in cells if cycle in errors[cell]
            }

        for window, groups in windows.items():
            pooled = [error for values in groups.values() for error in values]
            balanced = [mean(values) for values in groups.values()]
            soh_rows.append({
                **CONTEXT, "model": name, "window": window,
                "paired_test_cells": len(groups), "paired_cycles": len(pooled),
                "mae_pp": mean(balanced) if balanced else None,
                "worst_error_pp": max(pooled, default=None),
            })

        for cell in cells:
            direct_rul = direct[PRIOR][cell] if name == INCUMBENT else direct.get(name, {}).get(cell)
            projected_rul = timings[name][cell]
            actual_rul = endpoints[cell] - ANCHOR
            cell_rows.append({
                **CONTEXT, "cell_id": cell, "fold": test_folds[cell], "model": name,
                "observed_eol_cycle": endpoints[cell], "observed_rul_cycles": actual_rul,
                "raw_rul_prediction_cycles": direct_rul,
                "raw_rul_absolute_error_cycles": abs(direct_rul - actual_rul) if direct_rul is not None else None,
                "trajectory_rul_cycles": projected_rul,
                "trajectory_eol_signed_error_cycles": projected_rul - actual_rul if projected_rul is not None else None,
                "full_curve_mae_pp": mean(errors[cell].values()),
                "last_20_mae_pp": mean(tails[cell]),
                "last_20_worst_error_pp": max(tails[cell]),
                "observed_eol_soh_error_pp": errors[cell][endpoints[cell]],
                "feature_status": methods.get(name, {}).get(cell, "saved_reference"),
            })
    return soh_rows, cell_rows, [rul_metrics(name, predictions) for name, predictions in timings.items()]


direct_metrics = [rul_metrics(name, predictions) for name, predictions in direct.items()]
soh_rows, cell_rows, trajectory_rul_metrics = score_rollouts()
timing_by_model = {row["model"]: row for row in direct_metrics}
soh_by_window = {(row["model"], row["window"]): row for row in soh_rows}


def soh_mae(model, window):
    return soh_by_window[model, window]["mae_pp"]


def make_decisions():
    records = []
    for name in (CHARGE, DISCHARGE):
        complete = all(status == "fitted" for status in methods[name].values())
        rul_gain = timing_by_model[ENSEMBLE]["mae_cycles"] - timing_by_model[name]["mae_cycles"]
        full_gain = soh_mae(INCUMBENT, "full_curve") - soh_mae(name, "full_curve")
        tail_gain = soh_mae(INCUMBENT, "last_20_cycles") - soh_mae(name, "last_20_cycles")
        h10_delta = soh_mae(name, "horizon_10") - soh_mae(STATE, "horizon_10")
        h50_delta = soh_mae(name, "horizon_50") - soh_mae(STATE, "horizon_50")
        covers_rul = all(rollouts[name][cell]["rul"] is not None for cell in cells)
        passed = (complete and covers_rul and min(rul_gain, full_gain, tail_gain) > 0
                  and max(h10_delta, h50_delta) <= HORIZON_TOLERANCE)
        records.append({
            **CONTEXT, "model": name, "complete_feature_coverage": complete,
            "complete_trajectory_rul_coverage": covers_rul, "direct_rul_gain_cycles": rul_gain,
            "full_curve_gain_pp": full_gain, "last_20_gain_pp": tail_gain,
            "horizon_10_delta_pp": h10_delta, "horizon_50_delta_pp": h50_delta,
            "passes_benchmark": passed, "approved": False,
        })
    passing = [row for row in records if row["passes_benchmark"]]
    if passing:
        chosen = max(passing, key=lambda row: (row["last_20_gain_pp"], row["direct_rul_gain_cycles"]))
        chosen["approved"] = True
    return records


decisions = make_decisions()

In [7]:
def projection_exports():
    curves, rollout_starts = [], []
    for name in (SUMMARY, CHARGE, DISCHARGE):
        for cell in cells:
            item = rollouts[name][cell]
            common = {**CONTEXT, "cell_id": cell, "fold": test_folds[cell], "model": name}
            rollout_starts.append({
                **common, "initial_soh_pct": starts[cell][0], "event_status": "observed",
                "observed_eol_cycle": endpoints[cell], "predicted_rul_cycles": item["rul"],
                "forecast_status": "crossed" if item["rul"] is not None else "no_crossing_by_600",
                "input_rul_cycles": direct[name][cell], "transition_horizon_cycles": item["gate"],
                "target_duration_cycles": item["duration"], "feature_status": methods[name][cell],
            })
            for cycle, actual in sorted(observed[cell].items()):
                offset = cycle - ANCHOR
                curves.append({**common, "horizon_cycles": offset, "cycle": cycle,
                               "observed_soh_pct": actual, "predicted_soh_pct": item["values"][offset]})
    return curves, rollout_starts


def formatted(value):
    return f"{value:.3f}" if value is not None else "—"


def report_summary():
    lines = [
        "# XJTU 3C cycle-100 EOL timing",
        "",
        f"{len(cells)} verified cells across the five saved folds. Every feature comes from cycles through 100.",
        "",
        "## Direct RUL estimates",
        "",
        "| Model | Mean error (cycles) | Worst (cycles) | EOL ≤200 MAE (cycles) |",
        "| --- | ---: | ---: | ---: |",
    ]
    for row in direct_metrics:
        values = (row[key] for key in ("mae_cycles", "worst_error_cycles", "short_life_mae_cycles"))
        lines.append("| " + " | ".join([MODEL_NAMES[row["model"]], *map(formatted, values)]) + " |")
    lines.extend([
        "",
        "## SOH trajectories",
        "",
        "| Model | Full curve MAE (pp) | Last 20 MAE (pp) | Last 20 worst (pp) | +10 MAE (pp) | +50 MAE (pp) | Curve EOL MAE (cycles) |",
        "| --- | ---: | ---: | ---: | ---: | ---: | ---: |",
    ])
    for row in trajectory_rul_metrics:
        name = row["model"]
        values = [soh_mae(name, window) for window in ("full_curve", "last_20_cycles")]
        values.append(soh_by_window[name, "last_20_cycles"]["worst_error_pp"])
        values.extend((soh_mae(name, "horizon_10"), soh_mae(name, "horizon_50"), row["mae_cycles"]))
        lines.append("| " + " | ".join([MODEL_NAMES[name], *map(formatted, values)]) + " |")
    lines.extend(["", "## Dashboard decision", ""])
    selected = [MODEL_NAMES[row["model"]] for row in decisions if row["approved"]]
    if selected:
        lines.append(f"Approved **{selected[0]}** for 3C forecasts starting at cycle 100.")
    else:
        lines.append("Keep **Notebook 08 late-life calibrated** as the existing 3C dashboard model.")
    lines.extend([
        "",
        "Candidate approval requires complete coverage, lower direct RUL MAE than the equal-weight average, "
        "and lower full-curve and last-20 SOH MAE than Notebook 08. The +10 and +50 SOH errors may rise "
        "by at most 0.05 pp relative to the Notebook 07 SOH-dependent curve.",
        "",
        "All reported SOH MAEs weight cells equally. The per-cell table includes errors at the observed EOL. "
        "This is a within-cohort comparison on the same XJTU cells used for prior development.",
    ])
    return "\n".join(lines) + "\n"


curves, rollout_starts = projection_exports()
horizon_metrics = [
    {**CONTEXT, "model": row["model"], "horizon_cycles": int(row["window"].split("_")[-1]),
     "paired_test_cells": row["paired_test_cells"], "paired_mae_pp": row["mae_pp"]}
    for row in soh_rows if row["model"] in MODEL_FIELDS and row["window"].startswith("horizon_")
]
tables = {
    "ridge_parameters.csv": tuning,
    "direct_rul_metrics.csv": direct_metrics,
    "rul_metrics.csv": trajectory_rul_metrics,
    "soh_benchmark.csv": soh_rows,
    "soh_horizon_metrics.csv": horizon_metrics,
    "cell_metrics.csv": cell_rows,
    "model_selection.csv": decisions,
    "rollout_starts.csv": rollout_starts,
    "soh_trajectories.csv": curves,
}
for name, records in tables.items():
    write_csv(output / name, records)
summary = report_summary()
(output / "summary.md").write_text(summary, encoding="utf-8")
config = {
    "landmark_cycle": ANCHOR, "charge_rate_c": RATE, "eol_soh_pct": EOL_SOH,
    "cells": len(cells), "folds": sorted(training), "feature_windows": WINDOWS,
    "feature_version": FEATURE_VERSION, "ridge_alphas": ALPHAS, "horizon_tolerance_pp": HORIZON_TOLERANCE,
    "prior_reports": [NB06, NB07, NB08],
}
(output / "run_config.json").write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
archive_path = output.with_suffix(".zip")
with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    for name in (*tables, "curve_features.csv", "summary.md", "run_config.json"):
        archive.write(output / name, name)

try:
    from IPython import get_ipython
    from IPython.display import Markdown, display
except ImportError:
    print(summary)
else:
    if get_ipython() is None:
        print(summary)
    else:
        display(Markdown(summary))
print(f"Saved {archive_path}")

# XJTU 3C cycle-100 EOL timing

15 verified cells across the five saved folds. Every feature comes from cycles through 100.

## Direct RUL estimates

| Model | Mean error (cycles) | Worst (cycles) | EOL ≤200 MAE (cycles) |
| --- | ---: | ---: | ---: |
| Notebook 06 pooled Ridge | 21.937 | 52.848 | 21.900 |
| Notebook 06 recent-signal Ridge | 22.102 | 44.334 | 18.686 |
| Notebook 06 equal-weight average | 21.228 | 48.175 | 20.293 |
| Charge summary | 22.157 | 44.334 | 19.799 |
| Charge summary + charge curve | 24.866 | 60.736 | 17.649 |
| Charge summary + charge and discharge curves | 24.692 | 88.928 | 16.447 |

## SOH trajectories

| Model | Full curve MAE (pp) | Last 20 MAE (pp) | Last 20 worst (pp) | +10 MAE (pp) | +50 MAE (pp) | Curve EOL MAE (cycles) |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| Notebook 07 SOH-dependent fade | 1.576 | 2.627 | 8.758 | 0.241 | 0.396 | 23.600 |
| Notebook 08 late-life calibrated | 1.330 | 2.187 | 7.120 | 0.241 | 0.422 | 20.800 |
| Notebook 06 recent-signal Ridge | 1.415 | 2.212 | 7.372 | 0.241 | 0.438 | 19.600 |
| Notebook 06 equal-weight average | 1.349 | 2.141 | 7.238 | 0.241 | 0.428 | 20.133 |
| Charge summary | 1.428 | 2.243 | 7.372 | 0.241 | 0.446 | 20.200 |
| Charge summary + charge curve | 1.485 | 2.343 | 7.502 | 0.241 | 0.459 | 22.133 |
| Charge summary + charge and discharge curves | 1.537 | 2.553 | 9.421 | 0.241 | 0.459 | 23.467 |

## Dashboard decision

Keep **Notebook 08 late-life calibrated** as the existing 3C dashboard model.

Candidate approval requires complete coverage, lower direct RUL MAE than the equal-weight average, and lower full-curve and last-20 SOH MAE than Notebook 08. The +10 and +50 SOH errors may rise by at most 0.05 pp relative to the Notebook 07 SOH-dependent curve.

All reported SOH MAEs weight cells equally. The per-cell table includes errors at the observed EOL. This is a within-cohort comparison on the same XJTU cells used for prior development.


Saved /Users/sakibsmacbookm5/Documents/Python work/research-project/SICIP Final Project/reports/lifeguard_eol_timing_results.zip
